In [20]:
# In the previous lesson we built a RAG pipeline with RAGBase.rag() and saw it fail on the "Olama" typo. The search returned nothing useful, and the LLM had no way to recover.

# The flow that broke:

In [21]:
# The pipeline is fixed: search, build prompt, LLM.

# def rag(self, query):
#     search_results = self.search(query)
#     prompt = self.build_prompt(query, search_results)
#     answer = self.llm(prompt)
#     return answer

# The LLM is a passenger here, not a driver. It never sees the bad search results, so it can't try again with a corrected query.

In [22]:
# The difference is about who makes the decisions:

#     With RAG, the developer decides. We fix the steps up front, so search always runs once with the exact user query.
#     With an agent, the LLM decides. It chooses which actions to take and when to stop.

# The mechanism that makes this possible is function calling, and that's what the rest of this lesson is about.

In [23]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [24]:
MODEL = "openai/gpt-oss-20b"


In [25]:
# ASKING WITHOUT TOOLS,
message1 = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.chat.completions.create(
    model=MODEL,
    messages=message1,
)

response.choices[0].message.content

'Absolutely! If you’ve just found the course, you can still join—here’s how:\n\n1. **Log In / Sign Up**  \n   * If you already have an account, log in.  \n   * If you’re new, click the “Sign Up” button on the course page and fill in the quick registration form.  \n\n2. **Enroll**  \n   * Once logged in, you’ll see an “Enroll” button next to the course title. Click it to add the course to your dashboard.  \n\n3. **Check Availability**  \n   * If the class is full, you’ll see a “Waitlist” option. Enroll on the waitlist—if a spot opens, you’ll be notified automatically.\n\n4. **Payment (if required)**  \n   * Some courses are free, while others need a one‑time fee. After enrolling, you’ll be prompted to complete payment (credit card, PayPal, or your institution’s billing portal).\n\n5. **Confirmation**  \n   * You’ll receive a confirmation email with course start date, materials, and login details.  \n\n**Need help?**  \n* If you can’t find the course, let me know the course name or ID an

In [26]:
from ingest import load_faq_data, build_index

# Load the documents and build the index
docs = load_faq_data()
index = build_index(docs)


In [27]:
# Defining the tool

# First we define a top-level search function that queries the index directly. The model will reference it by this name. We keep the Python function and the tool name aligned so the dispatch is easier later.

def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [28]:


# Next we tell the model about this function. The model doesn't see our Python code, only a schema describing what the function does and what arguments it takes. LLMs are language agnostic. At the end we're just making an HTTP call, so we describe the tool in JSON rather than in Python. The same schema would work from TypeScript or Java.

search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
        }
    }
    
}

In [29]:
# Sending the question with the tool

# Now we send the same question as before, but this time we include the tool in the request:

response = openai_client.chat.completions.create(
    model=MODEL,
    messages=message1,
    tools=[search_tool],
)

response.choices[0].message.content

In [30]:
# Executing the function and sending the result back

# The function call contains JSON arguments. We parse them, call our search function, and serialize the result.

import json

tool_calls =response.choices[0].message.tool_calls

call = tool_calls[0].function

args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)


In [31]:
# Now we send this result back to the model. First, we add the model's output to the conversation history - the model needs to see its own function call. Then we add the tool result.

# 1. Add the assistant's message (which includes the tool call) to the history. 
# MUST be .append(), not .extend()
message1.append(response.choices[0].message)

# 2. Get the specific tool call object to retrieve its ID
tool_call = response.choices[0].message.tool_calls[0]

# 3. Append the tool's result to the history in the format OpenAI expects
message1.append({
    "role": "tool",
    "tool_call_id": tool_call.id,  # <-- The ID is here!
    "content": result_json
})


In [32]:
# Asking the model again

# We call the API a second time with the expanded history:

response = openai_client.chat.completions.create(
    model=MODEL,
    messages=message1,
    tools=[search_tool],
)

response.choices[0].message.content

'Absolutely! You can still jump into the course. The main thing to keep in mind is the project submission deadline:\n\n| What you want to do | Deadline | Notes |\n|--------------------|----------|-------|\n| **Join & participate** | Anytime before the start of the cohort | You’ll have full access to all videos, notebooks, and discussion channels. |\n| **Submit the capstone project** | Must be done while we’re still accepting submissions (usually before the end of the cohort’s grading period) | Required if you’d like to earn a certificate. If you miss that window, you’ll still get to work on the material, but the certificate won’t be available. |\n\n**How to get started**\n\n1. **Sign up / log in** to the course platform (e.g., [datatalks.club](https://courses.datatalks.club)).  \n2. **Browse the syllabus** and start with the first module.  \n3. **Check the project submission window** on the platform or in the cohort calendar.  \n4. **Drop your project** via the designated submission fo

In [33]:
# Token usage and cost

# We just made two API calls instead of one. Each call we send to the model costs money, so it's worth checking how much one tool-using turn actually costs.

# The response has a usage field with the token counts:

usage = response.usage
usage.input_tokens, usage.output_tokens

AttributeError: 'CompletionUsage' object has no attribute 'input_tokens'

In [ ]:
# For each model the provider publishes a price per million input tokens and per million output tokens. Plug those numbers in to convert tokens to dollars.


def calculate_model_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_model_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

In [ ]:
# # Conversationl history
# 1.Make a call to the LLM
# 2. LLM decided to invoke the search('params')
# 3. We invoke the search, we have the results 
# 4. send the results back to the LLM (another call)<--
# 5. LLM processes the results
# 6. LLM gives the answer